# 🎙️ Day-to-Day Multi-Agent Lab 2: Morning News Briefing & Voice Readout Assistant (LangGraph + TTS)
**Framework**: LangGraph (`StateGraph`, conditional edges, loop-back quality gate)  
**LLM Routing (from `.env`)**: **Anthropic (`ANTHROPIC_API_KEY` / `ANTHROPIC_MODEL`)** primary → **Ollama (`OLLAMA_BASE_URL` / `OLLAMA_FAST_MODEL`)** fallback. If both fail, raises the exact errors and remediation steps.  
**Tools**: Live RSS Feed Reader (`urllib` + `xml.etree.ElementTree`) + Voice Readout (`pyttsx3` / Windows `SpeechSynthesizer` + `IPython.display.Audio`)

### 📦 Step 0: Optional Dependency Installation
- **What**: Lists the packages used by this notebook (`langgraph`, `langchain-core`, `python-dotenv`, `pyttsx3`).
- **Why**: `langgraph` orchestrates the multi-step news pipeline, while `pyttsx3` (optional) provides cross-platform offline Text-to-Speech alongside Windows' built-in `SpeechSynthesizer`.
- **How**: Uncomment and run this cell once if any package is missing in `.venv`.

In [1]:
# Required packages (install into .venv if missing):
# %pip install -q langgraph langchain-core python-dotenv pyttsx3

### 🔐 Step 1: `.env` Loader, LLM Fallback Helper & News Feed Configuration
- **What**: Loads `.env` from the repository root, defines `invoke_llm_or_raise(prompt)` (Anthropic primary → Ollama fallback), and configures `USER_INTERESTS` and `RSS_FEEDS`.
- **Why**: Keeps API keys out of code, uses `OLLAMA_FAST_MODEL` as a fast local fallback when cloud APIs are unavailable, and lets you customize which topics and RSS feeds drive your morning briefing.
- **How**: `invoke_llm_or_raise` posts to Anthropic `/v1/messages` and falls back to Ollama `/api/generate` on failure, raising a detailed `RuntimeError` if both fail.

In [2]:
import os
import json
import urllib.request
import urllib.error
import xml.etree.ElementTree as ET
import subprocess
from pathlib import Path
from typing import TypedDict, List, Literal

def load_project_env() -> Path | None:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        env_file = parent / ".env"
        if env_file.is_file():
            try:
                from dotenv import load_dotenv
                load_dotenv(env_file, override=False)
            except ImportError:
                for line in env_file.read_text(encoding="utf-8").splitlines():
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
            return env_file
    return None

env_path = load_project_env()
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "").strip()
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-3-5-sonnet-20241022").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").strip()
OLLAMA_MODEL = os.getenv("OLLAMA_FAST_MODEL", os.getenv("OLLAMA_MODEL", "llama3.2:1b")).strip()

def invoke_llm_or_raise(prompt: str) -> str:
    """Try Anthropic API from .env first; fall back to local Ollama REST API; raise exact errors if both fail."""
    errors = []
    if ANTHROPIC_API_KEY:
        try:
            req = urllib.request.Request(
                "https://api.anthropic.com/v1/messages",
                data=json.dumps({
                    "model": ANTHROPIC_MODEL,
                    "max_tokens": 300,
                    "messages": [{"role": "user", "content": prompt}],
                }).encode("utf-8"),
                headers={
                    "x-api-key": ANTHROPIC_API_KEY,
                    "anthropic-version": "2023-06-01",
                    "content-type": "application/json",
                },
            )
            with urllib.request.urlopen(req, timeout=20) as resp:
                data = json.loads(resp.read().decode("utf-8"))
                print(f"🚀 Generated via Anthropic ({ANTHROPIC_MODEL})")
                return data["content"][0]["text"].strip()
        except urllib.error.HTTPError as e:
            body = e.read().decode("utf-8", errors="replace")
            errors.append(f"Anthropic ({ANTHROPIC_MODEL}) HTTP {e.code}: {body}")
        except Exception as e:
            errors.append(f"Anthropic ({ANTHROPIC_MODEL}) error: {type(e).__name__}: {e}")
        print(f"⚠️ {errors[-1]}\n   Falling back to Ollama ({OLLAMA_MODEL})...")
    else:
        errors.append("Anthropic skipped: ANTHROPIC_API_KEY is missing or empty in .env.")

    try:
        req = urllib.request.Request(
            f"{OLLAMA_BASE_URL.rstrip('/')}/api/generate",
            data=json.dumps({"model": OLLAMA_MODEL, "prompt": prompt, "stream": False}).encode("utf-8"),
            headers={"content-type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode("utf-8"))
            print(f"🦙 Generated via Ollama ({OLLAMA_MODEL})")
            return data["response"].strip()
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        errors.append(f"Ollama ({OLLAMA_MODEL} @ {OLLAMA_BASE_URL}) HTTP {e.code}: {body}")
    except Exception as e:
        errors.append(f"Ollama ({OLLAMA_MODEL} @ {OLLAMA_BASE_URL}) error: {type(e).__name__}: {e}")

    raise RuntimeError(
        "Both Anthropic and Ollama failed:\n  - " + "\n  - ".join(errors) +
        "\n\nHow to fix:\n"
        "1. Anthropic: Verify ANTHROPIC_API_KEY & ANTHROPIC_MODEL in .env and internet access/credits.\n"
        f"2. Ollama: Start Ollama (`ollama serve`) and ensure model is pulled (`ollama pull {OLLAMA_MODEL}`)."
    )

USER_INTERESTS = ["AI", "agents", "cloud", "economy", "technology"]
RSS_FEEDS = [
    ("HackerNews", "https://hnrss.org/frontpage?count=5"),
    ("BBC Tech", "https://feeds.bbci.co.uk/news/technology/rss.xml"),
]
print(f"🔐 .env: {env_path} | Primary: {ANTHROPIC_MODEL} | Fallback: {OLLAMA_MODEL}")

🔐 .env: d:\MyWork\Techacademy\UST-Global\.env | Primary: claude-sonnet-4-6 | Fallback: llama3.2:1b


### 🕸️ Step 2: LangGraph State, RSS Fetcher, Script Quality Gate & Voice Readout Nodes
- **What**: Defines `NewsBriefingState` and five graph components: `fetch_news_node`, `rank_stories_node`, `write_script_node`, `route_script_quality`, and `voice_readout_node`.
- **Why**: Raw RSS feeds are noisy and too long for audio. Ranking stories by keyword relevance, enforcing a ≤120-word brevity gate before TTS, and synthesizing `.wav` audio ensures a crisp, listenable 60-second briefing.
- **How**: `fetch_news_node` parses XML `<item><title>` tags from live RSS feeds; `rank_stories_node` scores headlines against `USER_INTERESTS`; `write_script_node` generates a radio-style script via the LLM; `route_script_quality` loops back once to tighten scripts >120 words; `voice_readout_node` writes `morning_briefing.wav` via `pyttsx3` or Windows `System.Speech`.

In [3]:
class NewsBriefingState(TypedDict):
    interests: List[str]
    raw_headlines: List[dict]
    ranked_stories: List[dict]
    spoken_script: str
    revision_count: int
    audio_path: str
    status: Literal["DRAFT", "READY_FOR_AUDIO", "DONE"]

def fetch_news_node(state: NewsBriefingState) -> dict:
    headlines = []
    feed_errors = []
    for source_name, url in RSS_FEEDS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=8) as resp:
                root = ET.fromstring(resp.read())
                for item in root.findall(".//item")[:4]:
                    title_el = item.find("title")
                    if title_el is not None and title_el.text:
                        headlines.append({"source": source_name, "title": title_el.text.strip()})
        except Exception as e:
            feed_errors.append(f"{source_name} ({url}): {type(e).__name__}: {e}")
    if not headlines:
        raise RuntimeError(
            "Failed to fetch any live RSS headlines:\n  - " + "\n  - ".join(feed_errors) +
            "\n\nHow to fix: Check internet/DNS/proxy connectivity or update RSS_FEEDS."
        )
    print(f"🌐 Fetched {len(headlines)} live headlines.")
    return {"raw_headlines": headlines}

def rank_stories_node(state: NewsBriefingState) -> dict:
    interests = [i.lower() for i in state["interests"]]
    scored = [{**h, "score": sum(1 for kw in interests if kw in h["title"].lower())} for h in state["raw_headlines"]]
    ranked = sorted(scored, key=lambda x: x["score"], reverse=True)[:3]
    print("⭐ Top 3 ranked stories selected.")
    return {"ranked_stories": ranked}

def write_script_node(state: NewsBriefingState) -> dict:
    rev = state.get("revision_count", 0)
    stories = state["ranked_stories"]
    bullets = "\n".join(f"- ({s['source']}) {s['title']}" for s in stories)
    prompt = (
        f"Write a natural, spoken-word morning radio news briefing under 90 words covering these 3 stories:\n{bullets}"
        if rev == 0 else
        f"Tighten this news briefing to strictly under 75 words for text-to-speech:\n{state['spoken_script']}"
    )
    script = invoke_llm_or_raise(prompt)
    print(f"📝 Script draft (rev {rev}, {len(script.split())} words):\n   {script}")
    return {"spoken_script": script, "revision_count": rev + 1}

def route_script_quality(state: NewsBriefingState) -> str:
    word_count = len(state["spoken_script"].split())
    if word_count > 120 and state["revision_count"] < 2:
        print(f"🔁 Quality Gate: {word_count} words > 120 limit, tightening script...")
        return "rewrite"
    return "voice"

def voice_readout_node(state: NewsBriefingState) -> dict:
    text = state["spoken_script"]
    out_wav = Path("morning_briefing.wav")
    tts_errors = []
    try:
        import pyttsx3
        engine = pyttsx3.init()
        engine.save_to_file(text, str(out_wav))
        engine.runAndWait()
        print(f"🔊 Generated voice readout with pyttsx3 -> {out_wav}")
        return {"audio_path": str(out_wav), "status": "DONE"}
    except Exception as e:
        tts_errors.append(f"pyttsx3: {type(e).__name__}: {e}")
    try:
        safe_text = text.replace("'", "''")
        ps_cmd = (
            "Add-Type -AssemblyName System.Speech; "
            "$s = New-Object System.Speech.Synthesis.SpeechSynthesizer; "
            f"$s.SetOutputToWaveFile('{out_wav.resolve()}'); "
            f"$s.Speak('{safe_text}'); $s.Dispose()"
        )
        subprocess.run(["powershell", "-NoProfile", "-Command", ps_cmd], check=True)
        print(f"🔊 Generated voice readout with Windows SpeechSynthesizer -> {out_wav}")
        return {"audio_path": str(out_wav), "status": "DONE"}
    except Exception as e:
        tts_errors.append(f"Windows SpeechSynthesizer: {type(e).__name__}: {e}")
    raise RuntimeError("TTS audio generation failed:\n  - " + "\n  - ".join(tts_errors) + "\nFix: Install pyttsx3 (`pip install pyttsx3`) or run on Windows with System.Speech.")

### 🔊 Step 3: Compile Graph, Run Briefing & Play Audio Inline
- **What**: Compiles the `StateGraph` with the conditional quality-gate loop, executes it on `initial_state`, and renders an interactive audio player in the notebook.
- **Why**: Demonstrates a complete end-to-end agentic pipeline—from live web RSS ingestion to LLM summarization, self-critique loop, and spoken-word audio output.
- **How**: Connects `START → fetch_news → rank_stories → write_script` with `add_conditional_edges("write_script", route_script_quality, ...)` and displays `morning_briefing.wav` using `IPython.display.Audio`.

In [4]:
try:
    from langgraph.graph import StateGraph, START, END
except ImportError as exc:
    raise RuntimeError(
        f"Missing dependency: {exc}\n"
        "Fix: Install LangGraph in your .venv:\n"
        "  .\\.venv\\Scripts\\python.exe -m pip install langgraph langchain-core python-dotenv pyttsx3"
    ) from exc

initial_state: NewsBriefingState = {
    "interests": USER_INTERESTS,
    "raw_headlines": [],
    "ranked_stories": [],
    "spoken_script": "",
    "revision_count": 0,
    "audio_path": "",
    "status": "DRAFT",
}

graph = StateGraph(NewsBriefingState)
graph.add_node("fetch_news", fetch_news_node)
graph.add_node("rank_stories", rank_stories_node)
graph.add_node("write_script", write_script_node)
graph.add_node("voice_readout", voice_readout_node)
graph.add_edge(START, "fetch_news")
graph.add_edge("fetch_news", "rank_stories")
graph.add_edge("rank_stories", "write_script")
graph.add_conditional_edges("write_script", route_script_quality, {"rewrite": "write_script", "voice": "voice_readout"})
graph.add_edge("voice_readout", END)
final_state = graph.compile().invoke(initial_state)

print("✅ Final Status:", final_state["status"], "| Audio:", final_state["audio_path"])
from IPython.display import Audio, display
display(Audio(final_state["audio_path"]))

🌐 Fetched 8 live headlines.
⭐ Top 3 ranked stories selected.
🚀 Generated via Anthropic (claude-sonnet-4-6)
📝 Script draft (rev 0, 89 words):
   Here's your morning briefing:

"Good morning! Staying with tech news this hour — the US is set to blacklist Asian AI company Megaspeed, following a scandal involving chip giant Nvidia. Speaking of Nvidia, just how vital are they to the American economy? Analysts say the answer might surprise you. And Jensen Huang — the man behind it all — has had quite the rollercoaster ride. From early struggles to becoming one of tech's biggest names, his story is one worth knowing. More on all of this at BBC Tech."
🔊 Generated voice readout with pyttsx3 -> morning_briefing.wav
✅ Final Status: DONE | Audio: morning_briefing.wav
